# 13 — Auditoria: Correlação Cruzada e Categóricas (treino vs. Kaggle)
## PRT Seguros

No notebook `07` vimos que a média/desvio-padrão/taxa de nulos das features mais "delatoras" são
quase idênticas entre treino e Kaggle — então o shift (AUC adversarial 0.72) não está em nenhuma
feature isolada. Duas hipóteses seguintes:

1. **Correlação entre pares de features diferente** entre as duas bases (a "forma da nuvem de pontos"
   muda mesmo que cada eixo isolado pareça igual)
2. **Alguma coluna categórica com bug de encoding** parecido com o da `regiao_Centro-Oeste`, mas em
   `segmento_*`, `veiculo_*`, `canal_*`, `metodo_pagamento_*` ou `escolaridade_*` — que só não apareceu
   antes porque olhamos só as top-10 features da validação adversarial (que são todas numéricas).


## 1. Carregar dados

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train = pd.read_csv("dados_processados/train_model_ready.csv")
val = pd.read_csv("dados_processados/val_model_ready.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready.csv")

ID_COL, TARGET_COL = "cod_individuo", "churned"
treino_completo = pd.concat([train, val], ignore_index=True)

feature_cols = [c for c in kaggle.columns if c != ID_COL]
numericas = [c for c in feature_cols if treino_completo[c].nunique() > 2]
categoricas = [c for c in feature_cols if c not in numericas]

print(f"Features numéricas/contínuas: {len(numericas)}")
print(f"Features binárias/dummy (0/1): {len(categoricas)}")


Features numéricas/contínuas: 26
Features binárias/dummy (0/1): 46


## 2. Correlação entre pares de features: treino vs. Kaggle

Calculamos a matriz de correlação de Pearson entre todas as features numéricas em cada base, e
olhamos a **diferença absoluta** entre as duas matrizes. Pares com diferença grande são candidatos a
explicar o shift "conjunto" que a comparação de médias não pegou.

In [2]:
corr_treino = treino_completo[numericas].corr()
corr_kaggle = kaggle[numericas].corr()

diff_corr = (corr_treino - corr_kaggle).abs()
diff_values = diff_corr.values.copy()
np.fill_diagonal(diff_values, 0)  # ignora a diagonal (sempre 1.0)
diff_corr = pd.DataFrame(diff_values, index=diff_corr.index, columns=diff_corr.columns)

# pega os pares (i, j) com maior diferença, sem duplicar (i,j) e (j,i)
pares = []
for i in range(len(numericas)):
    for j in range(i + 1, len(numericas)):
        pares.append((numericas[i], numericas[j], diff_corr.iloc[i, j]))

pares_df = pd.DataFrame(pares, columns=["feature_1", "feature_2", "diff_correlacao"])
pares_df = pares_df.sort_values("diff_correlacao", ascending=False)

print("Top 15 pares de features com maior diferença de correlação entre treino e Kaggle:")
top15 = pares_df.head(15).copy()
top15["corr_treino"] = top15.apply(lambda r: corr_treino.loc[r.feature_1, r.feature_2], axis=1)
top15["corr_kaggle"] = top15.apply(lambda r: corr_kaggle.loc[r.feature_1, r.feature_2], axis=1)
print(top15.to_string(index=False))


Top 15 pares de features com maior diferença de correlação entre treino e Kaggle:
                      feature_1                 feature_2  diff_correlacao  corr_treino  corr_kaggle
             tempo_cliente_dias     indice_relacionamento         0.016143     0.479118     0.462975
            num_apolices_ativas        tempo_cliente_dias         0.014474    -0.546130    -0.531656
             tempo_cliente_dias  num_produtos_contratados         0.013059     0.528049     0.514990
             tempo_cliente_dias     desconto_aplicado_pct         0.012939    -0.512569    -0.499631
        num_sinistros_historico  ultimo_login_portal_dias         0.012399     0.005421    -0.006978
                qtd_dependentes   num_sinistros_historico         0.010532    -0.002545     0.007987
                    renda_anual   num_sinistros_historico         0.010304    -0.004169     0.006135
       num_ligacoes_suporte_12m score_engajamento_digital         0.009935    -0.004512     0.005423
         

## 3. Auditoria de TODAS as colunas categóricas (mesmo teste que achou o bug da `regiao`)

Para cada grupo de dummies (`segmento_*`, `veiculo_*`, `canal_*`, `canal_aquisicao_*`,
`metodo_pagamento_*`, `escolaridade_*`, `tipo_cobertura_*`), comparamos:
- proporção de cada categoria no treino vs. Kaggle (shift de encoding, tipo o bug da região)
- taxa de churn dentro de cada categoria (se carrega sinal preditivo — se não, é candidata a remover
  caso tenha shift)

In [3]:
GRUPOS = {
    "segmento": [c for c in categoricas if c.startswith("segmento_")],
    "veiculo": [c for c in categoricas if c.startswith("veiculo_")],
    "canal_aquisicao": [c for c in categoricas if c.startswith("canal_aquisicao_")],
    "canal": [c for c in categoricas if c.startswith("canal_") and not c.startswith("canal_aquisicao_")],
    "metodo_pagamento": [c for c in categoricas if c.startswith("metodo_pagamento_")],
    "escolaridade": [c for c in categoricas if c.startswith("escolaridade_")],
    "tipo_cobertura": [c for c in categoricas if c.startswith("tipo_cobertura_")],
    "regiao": [c for c in categoricas if c.startswith("regiao_")],
}

linhas = []
for grupo, cols in GRUPOS.items():
    for c in cols:
        prop_treino = treino_completo[c].mean()
        prop_kaggle = kaggle[c].mean()
        taxa_churn = treino_completo.loc[treino_completo[c] == 1, TARGET_COL].mean() if treino_completo[c].sum() > 0 else np.nan
        linhas.append({
            "grupo": grupo, "coluna": c,
            "prop_treino": prop_treino, "prop_kaggle": prop_kaggle,
            "diff_absoluta": abs(prop_treino - prop_kaggle),
            "taxa_churn": taxa_churn,
        })

auditoria = pd.DataFrame(linhas).sort_values("diff_absoluta", ascending=False)
taxa_geral = treino_completo[TARGET_COL].mean()
auditoria["desvio_churn_vs_geral"] = (auditoria["taxa_churn"] - taxa_geral).abs()
auditoria


,grupo,coluna,prop_treino,prop_kaggle,diff_absoluta,taxa_churn,desvio_churn_vs_geral
18,canal,canal_Nao Informado,0.05767,0.02788,0.02979,0.123981,0.003041
7,veiculo,veiculo_NaN,0.04804,0.02412,0.02392,0.120733,0.000207
2,segmento,segmento_NaN,0.04653,0.02373,0.02280,0.119278,0.001662
35,regiao,regiao_Centro-Oeste,0.33553,0.33140,0.00413,0.120496,0.000444
32,tipo_cobertura,tipo_cobertura_basica,0.30019,0.29639,0.00380,0.194543,0.073603
33,tipo_cobertura,tipo_cobertura_padrao,0.33716,0.34084,0.00368,0.112232,0.008708
39,regiao,regiao_Sul,0.15153,0.15440,0.00287,0.119052,0.001888
25,metodo_pagamento,metodo_pagamento_pix,0.13290,0.13561,0.00271,0.119488,0.001452
29,escolaridade,escolaridade_Pos,0.11171,0.11388,0.00217,0.123266,0.002326
12,canal_aquisicao,canal_aquisicao_Digital,0.26733,0.26943,0.00210,0.123368,0.002428


## 4. Sinalizar candidatas a bug: shift alto E baixo poder preditivo

Mesmo critério do notebook `09`: só vale remover se o shift for grande **e** a coluna não ajudar a
prever churn (`desvio_churn_vs_geral` pequeno = a taxa de churn dentro da categoria é parecida com a
taxa geral, ou seja, a coluna não separa bem quem dá churn de quem não dá).

In [4]:
LIMIAR_SHIFT = 0.03       # 3 pontos percentuais de diferença de proporção
LIMIAR_PREDITIVO = 0.02   # desvio de taxa de churn menor que isso = pouco preditivo

suspeitas = auditoria[
    (auditoria["diff_absoluta"] > LIMIAR_SHIFT) &
    (auditoria["desvio_churn_vs_geral"] < LIMIAR_PREDITIVO)
]

if len(suspeitas) == 0:
    print("Nenhuma coluna categórica nova encontrada com shift alto E baixo poder preditivo.")
    print("O bug da 'regiao_Centro-Oeste' (já corrigido) parece ter sido o único desse tipo.")
else:
    print("Colunas candidatas a remover (shift alto, pouco preditivas):")
    print(suspeitas[["grupo", "coluna", "prop_treino", "prop_kaggle", "diff_absoluta", "taxa_churn"]])


Nenhuma coluna categórica nova encontrada com shift alto E baixo poder preditivo.
O bug da 'regiao_Centro-Oeste' (já corrigido) parece ter sido o único desse tipo.


## 5. Conclusão

- Se a célula 2 mostrar diferenças de correlação pequenas (< 0.05-0.10) em todos os pares, a hipótese
  de "shift na distribuição conjunta simples" também fica descartada — sobra principalmente a
  hipótese de artefato de geração/tratamento mais sutil (que o ajuste transdutivo do notebook `14`
  ataca de forma indireta, sem precisar identificar a causa exata).
- Se a célula 4 encontrar candidatas novas, elas devem ser adicionadas à lista de features removidas
  no próximo pipeline (`14`).
